<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Colab_15_Original_20B_Method_and_Input_Preflight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
import os, json, hashlib, re, ast, csv, collections, shutil
from datetime import datetime, timezone

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

MYDRIVE = Path(os.environ.get('ICHI2027_MYDRIVE', '/content/drive/MyDrive')).expanduser().resolve()
PROJECT = MYDRIVE / 'NeuroFHIR_ICHI2027'
OUT = PROJECT / 'phase15_outputs'
PRIVATE = OUT / '_PRIVATE_DO_NOT_SHARE'
PRIVATE.mkdir(parents=True, exist_ok=True)
SHAREABLE = OUT / 'ICHI2027_Phase15_SHAREABLE.json'
assert MYDRIVE.is_dir(), 'Please mount Google Drive before continuing.'

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def source(cell):
    s = cell.get('source', '')
    return ''.join(s) if isinstance(s, list) else str(s)

def verified_drive_path(rel):
    if not isinstance(rel, str) or not rel.strip():
        return None
    p = (MYDRIVE / rel).resolve()
    if not p.is_relative_to(MYDRIVE):
        return None
    return p if p.is_file() else None


Mounted at /content/drive


In [2]:
prior_candidates = [
    PROJECT / 'phase14_outputs' / 'ICHI2027_Phase14_SHAREABLE.json',
    Path('/content/ICHI2027_Phase14_SHAREABLE.json'),
]
if os.environ.get('ICHI2027_PHASE14_JSON'):
    prior_candidates.insert(0, Path(os.environ['ICHI2027_PHASE14_JSON']))
PRIOR_PATH = next((p for p in prior_candidates if p.is_file()), None)
if PRIOR_PATH is None:
    from google.colab import files
    selection = files.upload()
    name = 'ICHI2027_Phase14_SHAREABLE.json'
    if name not in selection:
        raise FileNotFoundError('Select the exact Phase 14 SHAREABLE JSON; do not upload private data.')
    PRIOR_PATH = Path('/content') / name
    PRIOR_PATH.write_bytes(selection[name])
phase14 = json.loads(PRIOR_PATH.read_text(encoding='utf-8'))
assert phase14.get('phase') == '14_targeted_original_TRACE_artifact_recovery'
assert phase14.get('four_arm_comparator_scores') is None
PHASE14_SHA = sha256_file(PRIOR_PATH)
private_candidates = [PROJECT / 'phase14_outputs' / 'ICHI2027_Phase14_PRIVATE_TRACE_RECOVERY.json']
if os.environ.get('ICHI2027_PHASE14_PRIVATE'):
    private_candidates.insert(0, Path(os.environ['ICHI2027_PHASE14_PRIVATE']))
PRIVATE14_PATH = next((p for p in private_candidates if p.is_file()), None)
previous_private = json.loads(PRIVATE14_PATH.read_text(encoding='utf-8')) if PRIVATE14_PATH else {}
if previous_private.get('run_utc') != phase14.get('run_utc'):
    previous_private = {}
print({'phase14_verified': True, 'phase14_private_manifest_available': bool(previous_private)})


{'phase14_verified': True, 'phase14_private_manifest_available': True}


In [3]:
# Exact notebook filenames and checksums come from the user's preceding verified audit.
ALLOWED_PREFIXES = (
    '20B_NeuroFHIR_TRACE_REAL_Three_State_Gate_Calibration_CORRECTED',
    '20B_NeuroFHIR_TRACE_Corrected_Three_State_Gate_Calibration_SELF_CONTAINED',
    '20B_NeuroFHIR_TRACE_REAL_Three_State_Gate_Calibration',
    '23B_NeuroFHIR_TRACE_Final_FHIR_Integration_Reproducibility_FIXED',
)
previous = [r for r in previous_private.get('target_notebooks', [])
            if r.get('file_present') and any(r.get('filename','').startswith(s) for s in ALLOWED_PREFIXES)]
known = {(r.get('filename'),r.get('content_sha256')):r for r in previous}
for r in phase14.get('prior_notebook_inventory', {}).get('target_notebook_basenames', []):
    if any(r.startswith(s) for s in ALLOWED_PREFIXES) and not any(a == r for a,b in known):
        # Phase 14 public JSON gives the basename but not its digest; use Phase 13 only when available.
        pass
candidates = []
for r in previous:
    p = verified_drive_path(r.get('private_relative_path'))
    if p and p.suffix.lower() == '.ipynb' and sha256_file(p) == r.get('content_sha256'):
        candidates.append({'filename':r['filename'], 'sha256':r['content_sha256'], 'path':p,
                           'origin':'PHASE14_VERIFIED_PRIVATE_MANIFEST'})
if not candidates:
    # Restricted fallback: find only specifically named NeuroFHIR notebooks, then hash them.
    public_names = set(phase14.get('prior_notebook_inventory', {}).get('target_notebook_basenames', []))
    for cur, dirs, names in os.walk(MYDRIVE):
        dirs[:] = [d for d in dirs if d not in {'.git','__pycache__','.ipynb_checkpoints','node_modules'}]
        for name in names:
            if name in public_names and any(name.startswith(s) for s in ALLOWED_PREFIXES):
                p = Path(cur) / name
                candidates.append({'filename':name, 'sha256':sha256_file(p), 'path':p,
                                   'origin':'TARGETED_FILENAME_FALLBACK_SHA_NOT_ATTESTED_BY_PHASE14'})
# De-duplicate actual files; no inferred scientific ranking.
candidates = list({str(r['path']):r for r in candidates}.values())
assert candidates, 'Original targeted 20B/23B notebooks not found. Restore the Phase 14 PRIVATE manifest in Drive.'
print({'exact_target_notebooks_found':len(candidates),
       'phase14_checksum_verified':sum(r['origin']=='PHASE14_VERIFIED_PRIVATE_MANIFEST' for r in candidates)})


{'exact_target_notebooks_found': 8, 'phase14_checksum_verified': 8}


In [4]:
# Extract actual method structure from original code without running notebook cells or retaining old outputs.
IO_READ = {'read_csv','read_parquet','read_json','read_excel','load','open','read_table'}
IO_WRITE = {'to_csv','to_parquet','to_json','dump','save','write_text','savez','savez_compressed'}
SIGNALS = {
    'threshold_calibration':r'(?i)(calibrat|threshold|quantile|q90|q95|repeat[-_ ]?pair)',
    'heldout_evaluation':r'(?i)(held[-_ ]?out|development|cross[-_ ]?fit|subject[-_ ]?split)',
    'three_state_logic':r'(?i)(three[-_ ]?state|review_required|trusted|caution|unstable|trace_state)',
    'data_lineage':r'(?i)(rid|imageuid|loniuid|examdate|source[-_ ]?hash)',
    'simulated_data_signal_for_review':r'(?i)(mock[-_ ]?data|synthetic|np\.random|random\.seed|default_rng|dummy)',
    'network_or_shell_signal_for_review':r'(?i)(requests\.|urlretrieve|wget|curl|!pip|%pip|subprocess|os\.system|\.download\()',
}
SIG = {k:re.compile(v) for k,v in SIGNALS.items()}

def call_name(func):
    if isinstance(func,ast.Name): return func.id
    if isinstance(func,ast.Attribute): return func.attr
    return ''

def literal_arg(call):
    if call.args and isinstance(call.args[0],ast.Constant) and isinstance(call.args[0].value,str):
        return Path(call.args[0].value.replace('\\','/')).name
    return None

def inspect_original_notebook(record):
    obj = json.loads(record['path'].read_text(encoding='utf-8'))
    assert obj.get('nbformat') == 4 and isinstance(obj.get('cells'),list), 'Invalid prior notebook'
    all_code = [c for c in obj['cells'] if c.get('cell_type')=='code']
    functions, calls, code_cells = [], collections.Counter(), []
    literal_files = set()
    method_signals = collections.Counter()
    syntax_errors = 0
    old_cells_with_outputs = 0
    old_error_outputs = 0
    for notebook_index, c in enumerate(obj['cells']):
        if c.get('cell_type')!='code': continue
        text = source(c)
        if c.get('outputs'): old_cells_with_outputs += 1
        old_error_outputs += sum(o.get('output_type')=='error' for o in c.get('outputs', []))
        flags = [k for k,v in SIG.items() if v.search(text)]
        method_signals.update(flags)
        rec = {'original_cell_index_zero_based':notebook_index,
               'source_sha256':hashlib.sha256(text.encode()).hexdigest(),
               'has_ipython_magics_or_shell':any(s.lstrip().startswith(('!','%')) for s in text.splitlines()),
               'method_signal_labels':flags, 'imports':[], 'function_names':[], 'read_calls':[], 'write_calls':[]}
        parseable = '\n'.join(line for line in text.splitlines()
                              if not line.lstrip().startswith(('!','%')))
        try: tree=ast.parse(parseable)
        except (SyntaxError,ValueError):
            syntax_errors+=1
            code_cells.append(rec)
            continue
        for node in ast.walk(tree):
            if isinstance(node,(ast.Import,ast.ImportFrom)):
                if isinstance(node,ast.Import):
                    rec['imports'].extend(n.name.split('.')[0] for n in node.names)
                else:
                    rec['imports'].append((node.module or '').split('.')[0])
            if isinstance(node,(ast.FunctionDef,ast.AsyncFunctionDef)):
                funcs = [a.arg for a in node.args.posonlyargs+node.args.args+node.args.kwonlyargs]
                functions.append({'name':node.name,'args':funcs,'code_cell_index':notebook_index,
                    'body_sha256':hashlib.sha256(ast.dump(node,include_attributes=False).encode()).hexdigest()})
                rec['function_names'].append(node.name)
            if isinstance(node,ast.Call):
                method = call_name(node.func)
                if method: calls[method]+=1
                if method in IO_READ|IO_WRITE:
                    entry={'method':method, 'literal_basename':literal_arg(node)}
                    rec['read_calls' if method in IO_READ else 'write_calls'].append(entry)
            if isinstance(node,ast.Constant) and isinstance(node.value,str) and len(node.value)<350:
                for m in re.findall(r'(?i)[A-Za-z0-9_(). -]+\.(?:csv|tsv|parquet|json|zip)',node.value):
                    b=Path(m.strip()).name
                    if len(b)<=150: literal_files.add(b)
        rec['imports'] = sorted(set(x for x in rec['imports'] if x))
        code_cells.append(rec)
    return {'filename':record['filename'], 'original_notebook_sha256':record['sha256'],
            'phase14_sha_attested':record['origin']=='PHASE14_VERIFIED_PRIVATE_MANIFEST',
            'original_code_cells':len(all_code),'original_total_cells':len(obj['cells']),
            'original_output_bearing_cells':old_cells_with_outputs,
            'original_embedded_error_outputs':old_error_outputs,'syntax_error_cells_in_static_parser':syntax_errors,
            'method_signals_by_code_cell':dict(method_signals),'function_definitions':functions,
            'import_modules':sorted(set(y for x in code_cells for y in x['imports'])),
            'original_literal_artifact_basenames':sorted(literal_files),
            'static_call_counts':dict(calls), 'code_cells_PRIVATE':code_cells,
            'unexecuted_original_file_path_PRIVATE':str(record['path'].relative_to(MYDRIVE))}

method_reports = [inspect_original_notebook(r) for r in candidates]
print({'code_inspected_notebooks':len(method_reports),
       'method_function_definitions':sum(len(x['function_definitions']) for x in method_reports),
       'original_embedded_error_outputs':sum(x['original_embedded_error_outputs'] for x in method_reports)})


{'code_inspected_notebooks': 8, 'method_function_definitions': 40, 'original_embedded_error_outputs': 0}


In [5]:
# Compare corrected copies for code identity. Saved notebook outputs never count as proof of reproducibility.
def code_fingerprint(nb_path):
    obj=json.loads(nb_path.read_text(encoding='utf-8'))
    pieces=[]
    for cell in obj.get('cells',[]):
        if cell.get('cell_type')=='code':
            content=source(cell)
            pieces.append(content.replace('\r\n','\n').strip())
    return hashlib.sha256('\n# CELL BREAK\n'.join(pieces).encode()).hexdigest()

corrected = [r for r in candidates if r['filename'].startswith('20B_NeuroFHIR_TRACE_REAL_Three_State_Gate_Calibration_CORRECTED')]
self_contained = [r for r in candidates if r['filename'].startswith('20B_NeuroFHIR_TRACE_Corrected_Three_State_Gate_Calibration_SELF_CONTAINED')]
if corrected:
    selected = sorted(corrected,key=lambda r:(r['origin']!='PHASE14_VERIFIED_PRIVATE_MANIFEST',r['sha256']))[0]
elif self_contained:
    selected = sorted(self_contained,key=lambda r:(r['origin']!='PHASE14_VERIFIED_PRIVATE_MANIFEST',r['sha256']))[0]
else:
    raise FileNotFoundError('Phase 15 needs at least one original 20B corrected or self-contained notebook.')
code_fingerprints={r['sha256']:code_fingerprint(r['path']) for r in candidates}
corrected_code_identical=(len({code_fingerprints[r['sha256']] for r in corrected})==1) if len(corrected)>1 else None
selected_report=next(x for x in method_reports if x['original_notebook_sha256']==selected['sha256'])
print({'selected_method_basename':selected['filename'],
       'selected_sha_phase14_verified':selected_report['phase14_sha_attested'],
       'corrected_versions_have_identical_code':corrected_code_identical})


{'selected_method_basename': '20B_NeuroFHIR_TRACE_REAL_Three_State_Gate_Calibration_CORRECTED.ipynb', 'selected_sha_phase14_verified': True, 'corrected_versions_have_identical_code': True}


In [6]:
# Make a PRIVATE copy of original code with all recorded outputs removed. Code itself is UNREDACTED:
# review paths/tokens before sharing it outside your private Drive. Do not execute it automatically.
try:
    import nbformat
except ImportError as exc:
    raise ImportError('nbformat is required. In Colab use: %pip install -q nbformat') from exc
original=json.loads(selected['path'].read_text(encoding='utf-8'))
copy_nb=nbformat.v4.new_notebook()
copy_nb['metadata']={'kernelspec':{'display_name':'Python 3','language':'python','name':'python3'},
                     'ichi2027_origin_sha256':selected['sha256'],
                     'ichi2027_note':'UNEXECUTED ORIGINAL CODE ONLY. PRIVATE. May contain hardcoded paths or tokens.'}
for original_index, c in enumerate(original['cells']):
    if c.get('cell_type')=='code':
        new_cell=nbformat.v4.new_code_cell(source=source(c), execution_count=None, outputs=[])
        new_cell['metadata']={'original_notebook_cell_index_zero_based':original_index}
        copy_nb['cells'].append(new_cell)
nbformat.validate(copy_nb)
COPY=PRIVATE/'ICHI2027_Phase15_Original_20B_CODE_ONLY_PRIVATE_REVIEW_BEFORE_SHARING.ipynb'
nbformat.write(copy_nb,str(COPY))
assert all(not c.get('outputs') for c in copy_nb['cells'])
print({'original_code_only_private_copy_created':True,'code_cells':len(copy_nb['cells']),
       'saved_outputs_removed':sum(bool(c.get('outputs')) for c in original['cells'] if c.get('cell_type')=='code')})


{'original_code_only_private_copy_created': True, 'code_cells': 21, 'saved_outputs_removed': 21}


In [7]:
# Optional authentic-input preflight. No synthetic MRIs or fabricated TRACE labels are created.
REAL_LONG = 'TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv'
REAL_REPEAT = 'TRACE_ADNI1_repeat_pairs_REAL.csv'
NEEDED = [REAL_LONG, REAL_REPEAT]
found_files=collections.defaultdict(list)
for current,dirs,names in os.walk(MYDRIVE):
    dirs[:]=[d for d in dirs if d not in {'.git','__pycache__','.ipynb_checkpoints','node_modules'}]
    for name in NEEDED:
        if name in names: found_files[name].append(Path(current)/name)
    if all(found_files.get(n) for n in NEEDED): break
for name in NEEDED:
    local=Path('/content')/name
    if local.is_file(): found_files[name].insert(0,local)
PROMPT_FOR_MISSING_REAL_INPUTS=os.environ.get('ICHI2027_PROMPT_UPLOAD','1')=='1'
missing = [n for n in NEEDED if not found_files.get(n)]
if missing and PROMPT_FOR_MISSING_REAL_INPUTS:
    try:
        from google.colab import files
        print('For the original 20B authentic-input preflight, select ONLY these real files:',missing)
        uploaded=files.upload()
        for name in missing:
            if name in uploaded:
                p=Path('/content')/name
                p.write_bytes(uploaded[name])
                found_files[name].append(p)
    except ImportError:
        pass
selected_inputs = {name:paths[0] for name,paths in found_files.items() if paths}
print({'real_input_files_located':sorted(selected_inputs),
       'real_input_files_missing':sorted(set(NEEDED)-set(selected_inputs))})


{'real_input_files_located': ['TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv', 'TRACE_ADNI1_repeat_pairs_REAL.csv'], 'real_input_files_missing': []}


In [8]:
# Verify real-input headers, numeric fields, cohort linkage and the earlier phase09 source digest.
FEATURES=['LHIPPOC','RHIPPOC','LENTORHIN','RENTORHIN','LMIDTEMP','RMIDTEMP','VENTRICLES','ICV']
phase09_private=PROJECT/'_private_reproducibility'/'phase09_private_reproducibility_manifest.json'
phase09_reference=json.loads(phase09_private.read_text(encoding='utf-8')) if phase09_private.is_file() else {}
reference_digest=phase09_reference.get('source_csv_sha256_private')
private_input_summary={}
for name,path in selected_inputs.items():
    issues=[]
    rows=0; ids=set(); numeric_present=0; nonblank_image_metadata=0
    with path.open(encoding='utf-8-sig',errors='replace',newline='') as stream:
        rd=csv.DictReader(stream)
        header=rd.fieldnames or []
        expected=({'RID','BL_EXAMDATE','M12_EXAMDATE'}|{f'{p}_{f}' for p in ['BL','M12'] for f in FEATURES}) if name==REAL_LONG else (
            {'RID','VISCODE','EXAMDATE','REPEAT_TYPE','IMAGEUID_1','IMAGEUID_2','LONIUID_1','LONIUID_2'}|
            {f'{f}_{i}' for f in FEATURES for i in [1,2]}|
            {f'DIFF_{f}' for f in FEATURES}|
            {f'PCT_DIFF_{f}' for f in FEATURES})
        missing_cols=sorted(expected-set(header))
        if missing_cols: issues.append('EXPECTED_COLUMNS_MISSING')
        for rec in rd:
            rows+=1
            rid=str(rec.get('RID','')).strip()
            if rid: ids.add(rid)
            else: issues.append('EMPTY_RID')
            if name==REAL_LONG:
                for prefix in ('BL','M12'):
                    for feature in FEATURES:
                        value=str(rec.get(f'{prefix}_{feature}','')).strip()
                        if value:
                            try:
                                from decimal import Decimal
                                if not Decimal(value).is_finite(): raise ValueError
                                numeric_present+=1
                            except (ValueError,ArithmeticError): issues.append('INVALID_NUMERIC_FEATURE')
            else:
                nonblank_image_metadata+=int(bool(rec.get('IMAGEUID_1')) and bool(rec.get('IMAGEUID_2')))
    digest=sha256_file(path)
    private_input_summary[name]={
        'path_PRIVATE':str(path), 'sha256_PRIVATE':digest,'rows':rows,'source_columns':len(header),
        'column_names_PRIVATE':header, 'unique_nonblank_RID':len(ids), 'numeric_events':numeric_present,
        'repeat_rows_with_both_image_ids':nonblank_image_metadata,
        'missing_required_columns':missing_cols, 'issues':sorted(set(issues)), 'all_RID_PRIVATE':sorted(ids),
        'phase09_source_sha_match':(digest==reference_digest) if (name==REAL_LONG and reference_digest) else None,
    }
if all(n in private_input_summary for n in NEEDED):
    longs=set(private_input_summary[REAL_LONG]['all_RID_PRIVATE'])
    repeats=set(private_input_summary[REAL_REPEAT]['all_RID_PRIVATE'])
    repeat_overlap=len(longs & repeats)
else: repeat_overlap=None
longrec=private_input_summary.get(REAL_LONG,{})
reprec=private_input_summary.get(REAL_REPEAT,{})
longitudinal_valid=(longrec.get('rows')==258 and longrec.get('source_columns')==51
                    and longrec.get('unique_nonblank_RID')==258 and longrec.get('numeric_events')==4128
                    and len(longrec.get('missing_required_columns',[]))==0 and not longrec.get('issues'))
repeat_valid=(reprec.get('rows')==90 and reprec.get('source_columns')==51 and not reprec.get('issues') and not reprec.get('missing_required_columns')
              and reprec.get('repeat_rows_with_both_image_ids')==90)
# An exact hash is available from Phase09 only for the longitudinal source, not repeat pairs.
print({'longitudinal_schema_and_numeric_audit_pass':longitudinal_valid,
       'repeat_pairs_schema_and_image_id_audit_pass':repeat_valid,
       'phase09_longitudinal_digest_reconciled':longrec.get('phase09_source_sha_match'),
       'repeat_participants_in_longitudinal':repeat_overlap})


{'longitudinal_schema_and_numeric_audit_pass': True, 'repeat_pairs_schema_and_image_id_audit_pass': True, 'phase09_longitudinal_digest_reconciled': True, 'repeat_participants_in_longitudinal': 67}


In [9]:
# Produce a reproducibility decision rather than treating old notebook outputs as current evidence.
selected_code_text='\n'.join(source(c) for c in original.get('cells',[]) if c.get('cell_type')=='code')
method_mentions_both_sources=(REAL_LONG in selected_code_text and REAL_REPEAT in selected_code_text)
source_sha_gate=(longrec.get('phase09_source_sha_match') is True)
input_gate=(longitudinal_valid and repeat_valid and source_sha_gate)
code_gate=(selected_report['phase14_sha_attested'] and selected_report['syntax_error_cells_in_static_parser']==0)
# Execution is deliberately NOT attempted. The original notebook may include live-drive and external calls.
preflight={
    'original_corrected_20b_notebook_selected':selected['filename'].startswith('20B_NeuroFHIR_TRACE_REAL_Three_State_Gate_Calibration_CORRECTED'),
    'original_notebook_hash_attested_by_phase14':selected_report['phase14_sha_attested'],
    'exact_original_code_only_private_copy_created':COPY.is_file(),
    'original_selected_notebook_code_parses_without_errors':selected_report['syntax_error_cells_in_static_parser']==0,
    'selected_notebook_statically_mentions_both_real_inputs':method_mentions_both_sources,
    'longitudinal_input_validated':longitudinal_valid,
    'repeat_input_validated':repeat_valid,
    'phase09_source_hash_available':bool(reference_digest),
    'phase09_longitudinal_source_hash_reconciled':source_sha_gate,
    'both_real_inputs_verified_against_available_manifest':input_gate,
    'ready_for_manual_review_before_original_20b_replay':bool(code_gate and input_gate and method_mentions_both_sources),
    'original_20b_method_was_executed_here':False,
    'calibration_parameters_verified_from_original_run':False,
    'original_TRACE_sidecar_recovered_or_regenerated':False,
    'official_micdm_load_completed':False,
    'genuine_four_arm_comparison_completed':False,
}


In [10]:
# Privacy separation: NEVER put original cell source, Drive paths, hashes of participant files,
# participant IDs, frozen outputs or calibration data in the shareable report.
run_utc=datetime.now(timezone.utc).isoformat()
private_report={
    'run_utc':run_utc,'phase14_shareable_sha256':PHASE14_SHA,
    'selected_notebook':{'filename':selected['filename'],'sha256':selected['sha256'],
                          'original_relative_path_PRIVATE':str(selected['path'].relative_to(MYDRIVE))},
    'original_method_static_reports_PRIVATE':method_reports,
    'code_only_private_copy_path':str(COPY.relative_to(MYDRIVE)),
    'code_only_private_copy_sha256':sha256_file(COPY),
    'all_available_input_evidence_PRIVATE':private_input_summary,
    'phase09_private_manifest_present':bool(reference_digest),
    'corrected_variant_code_fingerprints_PRIVATE':{r['sha256']:code_fingerprints[r['sha256']] for r in corrected},
    'gates':preflight,
}
(PRIVATE/'ICHI2027_Phase15_PRIVATE_METHOD_AND_INPUT_PREFLIGHT.json').write_text(
    json.dumps(private_report,indent=2,ensure_ascii=False),encoding='utf-8')
public={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'15_original_20b_method_extraction_and_authentic_input_preflight',
 'run_utc':run_utc,
 'status':'ACTUAL_ORIGINAL_NOTEBOOK_METHOD_STATIC_EXTRACTION_AND_REAL_INPUT_PREFLIGHT_NO_TRACE_REEXECUTION',
 'privacy':'Original code-only copy, exact source hashes, participant links and detailed method/code inspection stay PRIVATE in Google Drive.',
 'phase14_shareable_sha256':PHASE14_SHA,
 'original_method':{
    'target_original_notebooks_inspected':len(method_reports),
    'phase14_checksum_attested_target_notebooks':sum(x['phase14_sha_attested'] for x in method_reports),
    'corrected_20b_variants_inspected':len(corrected),
    'corrected_variants_code_identical':corrected_code_identical,
    'selected_original_notebook_basename':selected['filename'],
    'selected_original_code_cell_count':selected_report['original_code_cells'],
    'selected_original_function_names':sorted(set(f['name'] for f in selected_report['function_definitions']))[:80],
    'selected_original_method_signal_cell_counts':selected_report['method_signals_by_code_cell'],
    'selected_original_import_modules':selected_report['import_modules'],
    'selected_original_literal_trace_artifact_basenames':sorted(
        a for a in selected_report['original_literal_artifact_basenames']
        if re.search(r'(?i)(trace|gate|repeat|threshold|manifest|adni)',a))[:55],
    'original_executed_notebook_cell_outputs_present':selected_report['original_output_bearing_cells']>0,
    'original_notebook_error_outputs_count':selected_report['original_embedded_error_outputs'],
    'original_code_only_copy_created_PRIVATELY':COPY.is_file(),
    'old_notebook_outputs_removed_from_private_copy':True,
    'old_notebook_execution_not_attempted':True,
 },
 'authentic_input_preflight':{
    'longitudinal_source_present':REAL_LONG in selected_inputs,
    'repeat_pairs_source_present':REAL_REPEAT in selected_inputs,
    'longitudinal_rows':longrec.get('rows'),
    'longitudinal_exact_numeric_events':longrec.get('numeric_events'),
    'longitudinal_schema_and_values_pass':longitudinal_valid,
    'repeat_pairs_rows':reprec.get('rows'),
    'repeat_pairs_both_image_ids_count':reprec.get('repeat_rows_with_both_image_ids'),
    'repeat_pairs_schema_pass':repeat_valid,
    'repeat_participants_matched_to_longitudinal':repeat_overlap,
    'phase09_longitudinal_digest_comparison':('MATCH' if longrec.get('phase09_source_sha_match') is True else
         'MISMATCH' if longrec.get('phase09_source_sha_match') is False else 'REFERENCE_OR_INPUT_NOT_AVAILABLE'),
 },
 'gates':preflight,
 'original_TRACE_state_results':None,
 'four_arm_comparator_scores':None,
 'real_micdm_load_results':None,
 'external_evidence_still_required':['ORIGINAL_ADNI_UCSFFSL_EXPORT','FULL_ATHENA_VOCABULARY_PACKAGE'],
 'next_gate':('REVIEW_EXACT_20B_METHOD_CODE_THEN_REPLAY_ON_VERIFIED_ORIGINAL_REAL_INPUTS'
              if preflight['ready_for_manual_review_before_original_20b_replay'] else
              'RESTORE_EXACT_LONGITUDINAL_AND_REPEAT_CSVS_AND_PHASE09_MANIFEST_THEN_REVIEW_ORIGINAL_20B_CODE'),
}
SHAREABLE.write_text(json.dumps(public,indent=2,ensure_ascii=False),encoding='utf-8')
print({'phase15_shareable_saved':str(SHAREABLE),
       'private_method_code_only_copy':str(COPY),
       'ready_for_manual_review_before_original_replay':preflight['ready_for_manual_review_before_original_20b_replay'],
       'original_TRACE_outputs_recovered':False})


{'phase15_shareable_saved': '/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase15_outputs/ICHI2027_Phase15_SHAREABLE.json', 'private_method_code_only_copy': '/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase15_outputs/_PRIVATE_DO_NOT_SHARE/ICHI2027_Phase15_Original_20B_CODE_ONLY_PRIVATE_REVIEW_BEFORE_SHARING.ipynb', 'ready_for_manual_review_before_original_replay': True, 'original_TRACE_outputs_recovered': False}


In [11]:
try:
    from google.colab import files
    files.download(str(SHAREABLE))
except ImportError:
    print(SHAREABLE)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>